# LearnMate — M0 environment smoke test
Runs on a free Colab **T4** (Runtime → Change runtime type → GPU). Checkpoints and caches are written to Google Drive so a disconnect never loses work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -b claude/wonderful-bohr-1idsaa https://github.com/KirtirajChaudhari/LearnMate.git || (cd LearnMate && git pull)
%cd /content/LearnMate

In [ ]:
!pip -q install -e '.[gpu,eval,dev]'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# persist large/expensive artifacts on Drive
import os
D='/content/drive/MyDrive/learnmate'
for sub in ['data_raw','data_processed','data_cache','models','results']:
    os.makedirs(f'$D/{sub}', exist_ok=True)
!mkdir -p data && ln -sfn $D/data_raw data/raw && ln -sfn $D/data_processed data/processed && ln -sfn $D/data_cache data/cache && ln -sfn $D/models models && ln -sfn $D/results results

In [ ]:
!test -d data/raw/d2l-en || git clone https://github.com/d2l-ai/d2l-en.git data/raw/d2l-en
!git -C data/raw/d2l-en checkout -q 23d7a5aecceee57d1292c56e90cce307f183bb0a
!python -m learnmate.ingest --check
!python scripts/build_squad.py

Loads every model once, runs a forward pass and records latency and **peak VRAM** to `results/m0_smoke.json`. The fp16 vs NF4 result decides the frozen generator runtime (ADR-003).

In [ ]:
!python scripts/smoke_models.py
!cat results/m0_smoke.json

In [ ]:
!pip freeze > requirements.lock && head -50 requirements.lock

In [ ]:
!python -m pytest -q